# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umaimakhalid17/ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Lane 2 — Refresh / Content Opportunity Scoring** (locked in `w04_baseline_score`). Target:
`is_declining_label` (`trend_direction == "down"`) — the starter CSV's proxy label, same one
`w04`'s baseline was evaluated against.

## 1. Method choice and why

**Method: Logistic Regression first, then Random Forest** — both trained on the same features,
same split, same metric.

The question shape is "yes/no with an observed label" (`is_declining_label`), which the
training-honest-models skill maps straight to Logistic Regression → Random Forest: readable
first, stronger second, added only if it earns its complexity. Logistic regression also gives me
inspectable coefficients (Section 4), which matters more here than a marginal AUC gain — this
model has to be explainable to a content editor, not just accurate.

**Features — deliberately excluding the label-derived and product-flag columns:**
- Numeric: `search_volume`, `competition`, `cpc`, `word_count`, `char_count`, `impressions_90d`,
  `clicks_90d`, `sessions_90d`, `ai_sessions_90d`, `days_with_impressions`, `days_with_sessions`,
  `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`,
  `scroll_rate`, `ai_traffic_pct`
- Categorical: `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`,
  `word_count_tier`, `impression_tier`, `position_tier`
- `has_<col>` flags for the five columns with systematic missingness (`search_volume`,
  `competition`, `cpc`, `word_count`, `char_count`) — a blind `fillna(0)` on these would silently
  encode `content_type` into the features (per the flyrank-data skill's missingness gotcha),
  so the flag carries that information honestly instead.
- **Excluded on purpose:** `trend_direction`, `trend_pct` (these define the label — Section 3 of
  `w06` re-runs the exact leakage test on this final feature set). No FlyRank product flags
  (`health_score`, `priority_score`, `action_type`) exist in this dataset at all, so there's
  nothing to accidentally include there.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

np.random.seed(42)
pd.set_option("display.width", 140)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Rebuild the w04 baseline rule so it can be compared on the exact same rows / split below.
eligible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
ctr_gap = eligible & (df["ctr"] < 0.5)
severity = (0.5 - df["ctr"]).clip(lower=0)
df["baseline_action_score"] = np.where(ctr_gap, np.log1p(df["impressions_90d"]) * severity, 0.0)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
missingness_flag_cols = ["search_volume", "competition", "cpc", "word_count", "char_count"]

X = df[numeric_features + categorical_features].copy()
for c in missingness_flag_cols:
    X[f"has_{c}"] = df[c].notna().astype(int)
X[numeric_features] = X[numeric_features].fillna(0)
X[categorical_features] = X[categorical_features].fillna("unknown")
num_cols_final = numeric_features + [f"has_{c}" for c in missingness_flag_cols]

y = df["is_declining_label"].values
groups = df["client_id"].values

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols_final),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

print("Feature counts -- numeric (incl. has_ flags):", len(num_cols_final),
      "| categorical:", len(categorical_features))
print("Confirm no label-derived columns among features:",
      set(num_cols_final + categorical_features) & {"trend_direction", "trend_pct"})


Feature counts -- numeric (incl. has_ flags): 23 | categorical: 8
Confirm no label-derived columns among features: set()


## 2. Split design

**Grouped by `client_id` — not random.** Content items from the same client share hidden,
client-level character (the same SEO team, the same site authority, the same publishing
cadence) that correlates with several of my features. A random split lets rows from the same
client land in both train and test, so the model can partly "memorize" that client's baseline
behavior instead of learning a pattern that generalizes. The real deployment question is
**"does this work on a client the model has never seen?"** — `w06` Section 2 quantifies exactly
how much this choice matters by comparing this grouped split against a naive random split on
the same data.

In [2]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

print("train rows:", len(train_idx), "| test rows:", len(test_idx))
print("distinct clients -- train:", df.iloc[train_idx]["client_id"].nunique(),
      "| test:", df.iloc[test_idx]["client_id"].nunique())
overlap = set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"])
print("client_id overlap between train and test (must be 0 for a real grouped split):", len(overlap))

Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y[train_idx], y[test_idx]


train rows: 19166 | test rows: 10834
distinct clients -- train: 22 | test: 10
client_id overlap between train and test (must be 0 for a real grouped split): 0


## 3. Train + compare vs my baseline

Same test rows, same metric (**precision@K**, evaluated against `is_declining_label`), same base
rate reference as `w04`. `w04`'s baseline never used the label to build its score — it's a fair
comparison, not a rigged one.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

models = {}
for name, clf in [
    ("logistic_regression", LogisticRegression(max_iter=3000, random_state=42)),
    ("random_forest", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)),
]:
    pipe = Pipeline([("pre", preprocess), ("clf", clf)])
    pipe.fit(Xtr, ytr)
    proba = pipe.predict_proba(Xte)[:, 1]
    models[name] = {"pipe": pipe, "proba": proba, "auc": roc_auc_score(yte, proba)}

base_rate = yte.mean()
baseline_scores_test = df.iloc[test_idx]["baseline_action_score"].values

rows = []
for k in [50, 100, 250]:
    rows.append({
        "K": k,
        "base_rate": round(base_rate, 3),
        "baseline_rule (w04)": round(precision_at_k(baseline_scores_test, yte, k), 3),
        "logistic_regression": round(precision_at_k(models["logistic_regression"]["proba"], yte, k), 3),
        "random_forest": round(precision_at_k(models["random_forest"]["proba"], yte, k), 3),
    })
comparison_table = pd.DataFrame(rows)
print("AUC -- logistic_regression:", round(models["logistic_regression"]["auc"], 3),
      "| random_forest:", round(models["random_forest"]["auc"], 3))
print()
print("Precision@K on the SAME grouped-holdout rows, SAME metric as w04's baseline:")
print(comparison_table.to_string(index=False))


AUC -- logistic_regression: 0.604 | random_forest: 0.615

Precision@K on the SAME grouped-holdout rows, SAME metric as w04's baseline:
  K  base_rate  baseline_rule (w04)  logistic_regression  random_forest
 50      0.559                0.620                0.760          0.540
100      0.559                0.670                0.690          0.570
250      0.559                0.676                0.712          0.628


**Reading the table honestly:** logistic regression beats the baseline rule at every K tested
(0.76 vs 0.62 at K=50; 0.69 vs 0.67 at K=100; 0.712 vs 0.676 at K=250) — a real but modest lift,
not a blowout. Random forest is the more interesting finding: it's **worse than the baseline** at
K=50 (0.54 vs 0.62) and roughly tied at K=100, only pulling ahead at K=250. More model complexity
did not automatically buy more precision at the K values an editor actually works through first —
exactly the "add complexity only when it's earned" warning from the training-honest-models skill.
Logistic regression is my model going forward into `w06`/`w07`.

## 4. Errors and interpretation

What the winning model (logistic regression) actually leans on, and three concrete cases where
it's wrong.

In [4]:
cat_encoder = models["logistic_regression"]["pipe"].named_steps["pre"].named_transformers_["cat"]
feat_names = num_cols_final + list(cat_encoder.get_feature_names_out(categorical_features))
coefs = models["logistic_regression"]["pipe"].named_steps["clf"].coef_[0]
coef_table = (
    pd.DataFrame({"feature": feat_names, "coef": coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
)
print("Top 10 logistic regression coefficients (by |coefficient|):")
print(coef_table.head(10).to_string(index=False))
print()
print("Sanity check: no single feature towers over the rest the way a leaked column would")
print("(compare this to w06 Section 3, where deliberately adding a leaky column DOES tower).")


Top 10 logistic regression coefficients (by |coefficient|):
                  feature      coef
      position_tier_top_3 -1.161265
    days_with_impressions  0.846431
         content_age_days -0.538363
word_count_tier_1000-2000  0.485802
competition_level_unknown  0.424812
       days_with_sessions -0.408570
 main_intent_navigational -0.389277
       position_tier_deep  0.382473
      freshness_tier_181+ -0.370716
             avg_position -0.370368

Sanity check: no single feature towers over the rest the way a leaked column would
(compare this to w06 Section 3, where deliberately adding a leaky column DOES tower).


In [5]:
test_df = df.iloc[test_idx].copy().reset_index(drop=True)
test_df["model_proba"] = models["logistic_regression"]["proba"]
test_df["rank_by_model"] = test_df["model_proba"].rank(ascending=False, method="first")

wrong_fp = (
    test_df[(test_df["rank_by_model"] <= 100) & (test_df["is_declining_label"] == 0)]
    .sort_values("model_proba", ascending=False)
)
show_cols = ["content_id", "model_proba", "impressions_90d", "avg_position", "ctr",
             "content_age_days", "days_since_last_update", "word_count"]
print("3 wrong cases -- ranked in the model's top 100 by probability, but NOT actually declining:")
print(wrong_fp[show_cols].head(3).to_string(index=False))


3 wrong cases -- ranked in the model's top 100 by probability, but NOT actually declining:
          content_id  model_proba  impressions_90d  avg_position  ctr  content_age_days  days_since_last_update  word_count
content_374e795aab68     0.950434              235          31.0 0.85               181                      20      2675.0
content_7be5f150dc65     0.928680              290           5.9 0.00                96                      20      2481.0
content_41baf0722ad9     0.915155             3115          12.8 0.00               275                     104      1596.0


**Why these are hard, not just wrong:**

1. **`content_374e795aab68`** — CTR is actually decent (0.85%, above the 0.5% flag threshold)
   and position is deep (31.0). The model likely leans on `content_age_days` (181, right at the
   freshness-tier boundary) and `days_since_last_update` (20, recently touched) pulling in
   opposite directions — a genuinely ambiguous case, not an obvious model failure.
2. **`content_7be5f150dc65`** — `ctr = 0.00%` at a strong position (5.9) looks exactly like the
   pattern the model (and `w04`'s rule) both associate with decline, yet `trend_direction` came
   back non-"down" for this row. This is the proxy label's own noise: `trend_direction` is a
   30-day-vs-prior-30-day snapshot, and a page can have a rough CTR problem while still holding
   flat or slightly-up impressions in that particular window.
3. **`content_41baf0722ad9`** — very high impressions (3,115) but `ctr = 0.00%` and a large,
   104-day-stale gap; this looks like the strongest "should be declining" case of the three, and
   its being a false positive is the best argument for treating any single row's prediction as a
   starting point for a human look, not a verdict.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.